# Food-101 fast.ai Transfer Learning Walkthrough

This notebook is the interactive route for the transfer-learning chapter. It is intentionally thin: `food101_fastai.py` remains the canonical implementation for repeatable runs, metadata, metrics, and artifacts.

The default cells start in synthetic smoke mode so you can inspect the data pipeline, freezing behavior, and saved artifacts without downloading Food-101 or pretrained weights.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

## 1. Point at the Companion Code

Run this notebook from `chapter_transfer_learning_fastai/` when possible. If the current directory is the repository root, the fallback below still finds the script.

In [ ]:
from pathlib import Path
import csv
import json
import shlex
import subprocess
import sys

from IPython.display import Image, display


def find_chapter_dir(script_name: str, chapter_name: str) -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / chapter_name, base / "code" / chapter_name):
            if (candidate / script_name).exists():
                return candidate.resolve()
    raise FileNotFoundError(f"Could not locate {script_name}")


NOTEBOOK_DIR = find_chapter_dir("food101_fastai.py", "chapter_transfer_learning_fastai")
SCRIPT = NOTEBOOK_DIR / "food101_fastai.py"

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

RUNS_DIR = NOTEBOOK_DIR / "runs"
print("Notebook directory:", NOTEBOOK_DIR)
print("Canonical script:", SCRIPT)

## 2. Run a Dependency Check

This calls the companion script rather than reimplementing its environment check. If dependencies are missing, install the transfer-learning group from this directory with `poetry install --with transfer-learning` before continuing.

In [ ]:
check_cmd = [
    sys.executable,
    str(SCRIPT),
    "--check-deps",
    "--allow-missing-deps",
    "--output-dir",
    str(RUNS_DIR / "notebook-deps"),
]
print(" ".join(shlex.quote(part) for part in check_cmd))
subprocess.run(check_cmd, cwd=NOTEBOOK_DIR, check=True)

## 3. Configure a Small Interactive Run

Keep `SMOKE_MODE = True` while learning the workflow. Switch it off only when you intend to download Food-101 and use pretrained ImageNet weights.

In [ ]:
SMOKE_MODE = True
IMAGE_SIZE = 64 if SMOKE_MODE else 224
RESIZE_SIZE = 72 if SMOKE_MODE else 460
BATCH_SIZE = 4 if SMOKE_MODE else 64
SEED = 42
ARCHITECTURE = "resnet18" if SMOKE_MODE else "resnet34"
USE_PRETRAINED_WEIGHTS = False if SMOKE_MODE else True

RUN_INTERACTIVE_TRAINING = False

print({
    "smoke_mode": SMOKE_MODE,
    "architecture": ARCHITECTURE,
    "pretrained": USE_PRETRAINED_WEIGHTS,
    "image_size": IMAGE_SIZE,
    "resize_size": RESIZE_SIZE,
    "batch_size": BATCH_SIZE,
})

## 4. Build Data Loaders and Inspect a Batch

Before running this cell, predict where the label for each image comes from and which split remains reserved for final evaluation.

In [ ]:
import fastai.vision.all as fastai_vision
from fastai.vision.all import *
from torchvision import models as tv_models

from food101_fastai import (
    architecture_callable,
    architecture_weights,
    create_synthetic_images,
    food101_files,
    model_parameter_counts,
    set_reproducible_seed,
)

set_reproducible_seed(SEED)

if SMOKE_MODE:
    data_root = RUNS_DIR / "notebook-smoke-data"
    items = L(create_synthetic_images(data_root, classes=2, items_per_class=8, size=RESIZE_SIZE, seed=SEED))
    reserved_test_items = L([])
else:
    data_root = untar_data(URLs.FOOD)
    items = food101_files(fastai_vision, "train", Path(data_root))
    reserved_test_items = food101_files(fastai_vision, "test", Path(data_root))

food = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_y=parent_label,
    splitter=RandomSplitter(valid_pct=0.2, seed=SEED),
    item_tfms=Resize(RESIZE_SIZE),
    batch_tfms=aug_transforms(size=IMAGE_SIZE),
)

dls = food.dataloaders(items, bs=BATCH_SIZE, num_workers=0)
print("data root:", data_root)
print("train items:", len(dls.train_ds))
print("validation items:", len(dls.valid_ds))
print("reserved test items:", len(reserved_test_items))
print("vocab:", list(dls.vocab)[:10])

dls.show_batch(max_n=min(4, len(dls.train_ds)), figsize=(6, 6))

## 5. Build the Learner and Inspect Freezing

`vision_learner` creates the transfer-learning model. In smoke mode the architecture is the same kind of ResNet body, but the weights are random so the cell stays offline.

In [ ]:
arch = architecture_callable(tv_models, ARCHITECTURE)
weights = architecture_weights(tv_models, ARCHITECTURE, USE_PRETRAINED_WEIGHTS)

learn = vision_learner(
    dls,
    arch,
    pretrained=USE_PRETRAINED_WEIGHTS,
    weights=weights,
    metrics=accuracy,
)

print("after build:", model_parameter_counts(learn.model))
learn.freeze()
print("after freeze:", model_parameter_counts(learn.model))
learn.unfreeze()
print("after unfreeze:", model_parameter_counts(learn.model))
learn.freeze()

## 6. Optional In-Notebook Training

The script is still the preferred way to create reproducible artifacts. Use this cell only when you want to watch one tiny fit call from inside the notebook.

In [ ]:
if RUN_INTERACTIVE_TRAINING:
    learn.fit_one_cycle(1, slice(3e-3), pct_start=0.99)
else:
    print("Set RUN_INTERACTIVE_TRAINING = True to train this in-notebook learner.")

## 7. Run the Canonical Smoke Command

This cell shells out to `food101_fastai.py`. It is the same route used for repeatable command-line runs, so the artifacts match the README contract.

In [ ]:
smoke_output_dir = RUNS_DIR / "notebook-smoke-run"
smoke_cmd = [
    sys.executable,
    str(SCRIPT),
    "--smoke",
    "--architecture",
    "resnet18",
    "--freeze-epochs",
    "1",
    "--epochs",
    "0",
    "--image-size",
    "64",
    "--resize-size",
    "72",
    "--batch-size",
    "4",
    "--max-top-losses",
    "4",
    "--output-dir",
    str(smoke_output_dir),
]
print(" ".join(shlex.quote(part) for part in smoke_cmd))
subprocess.run(smoke_cmd, cwd=NOTEBOOK_DIR, check=True)

## 8. Read the Run Artifacts

The script writes JSON, CSV, and optional PNG evidence. These files are the reproducibility record you should cite in a report.

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def read_csv_rows(path):
    with Path(path).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


metadata = read_json(smoke_output_dir / "metadata.json")
metrics = read_json(smoke_output_dir / "metrics.json")
summary = {
    "mode": metadata["mode"],
    "split_rule": metadata["data_split_rule"],
    "reserved_test_rule": metadata["reserved_test_rule"],
    "status": metrics["status"],
    "valid_accuracy": metrics.get("valid_accuracy"),
    "test_evaluation": metrics.get("test_evaluation"),
}
print(json.dumps(summary, indent=2))

stage_metrics = smoke_output_dir / "stage_metrics.csv"
if stage_metrics.exists():
    print("stage metrics:")
    for row in read_csv_rows(stage_metrics):
        print(row)

top_losses = smoke_output_dir / "validation_top_losses.csv"
if top_losses.exists():
    print("top losses:")
    for row in read_csv_rows(top_losses):
        print(row)

top_loss_grid = smoke_output_dir / "validation_top_losses.png"
if top_loss_grid.exists():
    display(Image(filename=str(top_loss_grid)))

## 9. Full Food-101 Baseline Command

Leave this guarded until the dataset download and training time are intended. The official test split is still not evaluated here.

In [ ]:
RUN_FULL_FOOD101 = False
full_output_dir = RUNS_DIR / "resnet34-baseline"
full_cmd = [
    sys.executable,
    str(SCRIPT),
    "--architecture",
    "resnet34",
    "--image-size",
    "224",
    "--resize-size",
    "460",
    "--batch-size",
    "64",
    "--freeze-epochs",
    "1",
    "--epochs",
    "5",
    "--base-lr",
    "3e-3",
    "--output-dir",
    str(full_output_dir),
]
print(" ".join(shlex.quote(part) for part in full_cmd))

if RUN_FULL_FOOD101:
    subprocess.run(full_cmd, cwd=NOTEBOOK_DIR, check=True)
else:
    print("Review the command first. Set RUN_FULL_FOOD101 = True when you intend a full run.")

## 10. Reserved Test Evaluation

Only add `--evaluate-test` after selecting a model from validation evidence. Do not use this cell for repeated tuning.

In [ ]:
EVALUATE_RESERVED_TEST = False
final_test_cmd = full_cmd + ["--evaluate-test"]
print(" ".join(shlex.quote(part) for part in final_test_cmd))

if EVALUATE_RESERVED_TEST:
    subprocess.run(final_test_cmd, cwd=NOTEBOOK_DIR, check=True)
else:
    print("Leave EVALUATE_RESERVED_TEST = False until the final selected configuration.")

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.